# Step 2: Memory & Conversation Chains

**Resource:** [Conversational Memory in LangChain – Aurelio AI](https://www.aurelio.ai/learn/langchain-conversational-memory)

In this notebook we cover the **modern (non-deprecated)** approach to two memory types:

| # | Memory Type | Modern Implementation |
|---|-------------|----------------------|
| 1 | `ConversationBufferMemory` | `RunnableWithMessageHistory` |
| 2 | `ConversationBufferWindowMemory` | `RunnableWithMessageHistory` + custom history class |

> All code uses **ChatGroq** with `llama-3.3-70b-versatile` instead of OpenAI.

---
## Setup: Install & Import Dependencies

In [ ]:
# Install required packages (run once in Colab)
!pip install -q langchain langchain-groq langchain-core python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

# Load GROQ_API_KEY from .env file
# In Colab you can also use: from google.colab import userdata; os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
load_dotenv()

# Verify the key is loaded
assert os.getenv("GROQ_API_KEY"), "GROQ_API_KEY not found! Check your .env file."
print("API key loaded successfully.")

In [ ]:
from langchain_groq import ChatGroq

# Initialize the LLM — using Groq instead of OpenAI
# temperature=0 means deterministic, consistent responses (good for learning)
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.0
)

print("LLM ready:", llm.model_name)

---
## Section 1: `ConversationBufferMemory` with `RunnableWithMessageHistory`

### What is this?

**ConversationBufferMemory** is the simplest form of memory — it stores the **entire conversation history** from start to finish, without any compression or trimming.

Every message you send and every reply from the AI is saved and sent back to the model on each new turn. This lets the model "remember" everything said earlier in the conversation.

### Why `RunnableWithMessageHistory`?

The older `ConversationChain` class is deprecated. The modern way to add memory to a chain is by wrapping it with `RunnableWithMessageHistory`. Here's how it works:

```
User query
    ↓
RunnableWithMessageHistory  ← looks up session history
    ↓
ChatPromptTemplate          ← slots in: system prompt + history + new query
    ↓
ChatGroq LLM                ← generates a reply
    ↓
Response  →  history saved back
```

### Key concept: Session ID
A **session ID** is just a string label (like `"chat_001"`) that identifies a conversation. You can have many separate conversations running at once — each has its own isolated history.

In [ ]:
from langchain_core.prompts import (
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    MessagesPlaceholder,
    ChatPromptTemplate
)

# Define the system prompt — sets the AI's persona
system_prompt = "You are a helpful assistant called Zeta."

# Build the prompt template with three parts:
#   1. SystemMessage   — instructions / persona
#   2. history         — placeholder where past messages will be injected
#   3. HumanMessage    — the current user query
prompt_template = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(system_prompt),
    MessagesPlaceholder(variable_name="history"),  # memory goes here
    HumanMessagePromptTemplate.from_template("{query}"),
])

print("Prompt template created.")

In [ ]:
# Chain the prompt template and LLM together using LCEL (| operator)
# This is the core pipeline: prompt → LLM
pipeline = prompt_template | llm

print("Pipeline created: prompt_template | llm")

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory

# This dictionary acts as our in-memory "database" of conversations.
# Key = session_id (string), Value = InMemoryChatMessageHistory object
chat_map = {}

def get_chat_history(session_id: str) -> InMemoryChatMessageHistory:
    """Look up (or create) the chat history for a given session ID."""
    if session_id not in chat_map:
        # First time we see this session — create a fresh history
        chat_map[session_id] = InMemoryChatMessageHistory()
        print(f"  [New session created: '{session_id}']")
    return chat_map[session_id]

print("get_chat_history function defined.")

In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory

# Wrap the pipeline with message history management
# - get_session_history: function to fetch the right history by session ID
# - input_messages_key: the key in our invoke() dict that holds the user's query
# - history_messages_key: must match the MessagesPlaceholder variable name in the prompt
pipeline_with_history = RunnableWithMessageHistory(
    pipeline,
    get_session_history=get_chat_history,
    input_messages_key="query",
    history_messages_key="history"
)

print("RunnableWithMessageHistory created.")

In [ ]:
# --- Turn 1: Introduce ourselves ---
# We pass a session_id in config so the runnable knows which history to use
response1 = pipeline_with_history.invoke(
    {"query": "Hi, my name is Zariya!"},
    config={"configurable": {"session_id": "session_001"}}
)

print("AI:", response1.content)

In [ ]:
# --- Turn 2: Ask a follow-up ---
# Same session_id = AI has access to the previous turn
response2 = pipeline_with_history.invoke(
    {"query": "I am learning LangChain. What are the main components of LCEL?"},
    config={"configurable": {"session_id": "session_001"}}
)

print("AI:", response2.content)

In [ ]:
# --- Turn 3: Test memory ---
# The AI should remember our name from Turn 1
response3 = pipeline_with_history.invoke(
    {"query": "What is my name, and what am I learning?"},
    config={"configurable": {"session_id": "session_001"}}
)

print("AI:", response3.content)

In [ ]:
# Inspect what's stored in memory for session_001
# You can see every HumanMessage and AIMessage saved so far
print("--- Full conversation history for session_001 ---\n")
for msg in chat_map["session_001"].messages:
    role = "Human" if msg.__class__.__name__ == "HumanMessage" else "AI"
    print(f"[{role}]: {msg.content[:100]}..." if len(msg.content) > 100 else f"[{role}]: {msg.content}")
    print()

In [ ]:
# --- Bonus: Different session ID = completely separate memory ---
# This new session has no idea what happened in session_001
response_new = pipeline_with_history.invoke(
    {"query": "What is my name?"},
    config={"configurable": {"session_id": "session_002"}}
)

print("AI (session_002):", response_new.content)
# Expected: AI won't know the name — different session, empty history

---
### Practice Exercise 1

Test your understanding of `ConversationBufferMemory` by completing the tasks below.

**Tasks:**
1. Start a new session called `"my_session"` and tell the AI your favorite programming language.
2. In the same session, ask the AI to give you a fun fact about that language.
3. In the same session, ask the AI: *"What language did I tell you I liked?"* — verify it remembers.
4. Start a second session `"other_session"` and ask the same question — verify it does NOT remember.

**Reflection questions (answer in a new markdown cell):**
- What happens to memory when you restart the Python kernel?
- When would `ConversationBufferMemory` become a problem for very long conversations?

In [ ]:
# --- YOUR CODE HERE ---

# Task 1: Tell the AI your favorite language in 'my_session'


# Task 2: Ask for a fun fact in the same session


# Task 3: Ask if it remembers your language


# Task 4: Try the same question in 'other_session'


---
## Section 2: `ConversationBufferWindowMemory` with `RunnableWithMessageHistory`

### What is this?

**ConversationBufferWindowMemory** is like ConversationBufferMemory, but with a sliding window. It only keeps the **last `k` messages** and silently drops anything older.

For example, with `k=4` (4 messages = 2 user turns + 2 AI replies), the memory looks like this after a long conversation:

```
Older messages (dropped):  "Hi I'm Zariya"  →  "Hello Zariya!"
                           "I like Python"  →  "Great choice!"
Window (kept):             "What is LCEL?"  →  "LCEL is..."
                           "Give me an example" → "Sure, ..."
```

### Why would you use this?

| Benefit | Explanation |
|---------|-------------|
| **Cheaper** | Fewer tokens sent per request = lower API cost |
| **Faster** | Less context for the model to process = lower latency |
| **Better focus** | LLMs can lose focus on instructions when given too much context |
| **No context overflow** | Buffer memory will eventually exceed the model's max context window; window memory never will |

### The trade-off
The AI will **forget older messages**. If you introduce yourself early in a long conversation, the AI may not remember your name later.

### How we implement it
Since `RunnableWithMessageHistory` uses `InMemoryChatMessageHistory` which keeps everything, we create a **custom subclass** that trims to the last `k` messages after every update.

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage

class BufferWindowMessageHistory(BaseChatMessageHistory, BaseModel):
    """Custom chat history that keeps only the last k messages."""
    
    messages: list[BaseMessage] = Field(default_factory=list)
    k: int = Field(default_factory=int)  # max number of messages to keep

    def __init__(self, k: int):
        super().__init__(k=k)
        print(f"  [BufferWindowMessageHistory initialized with k={k}]")

    def add_messages(self, messages: list[BaseMessage]) -> None:
        """Add new messages, then trim to keep only the last k messages."""
        # First, add the new messages to the list
        self.messages.extend(messages)
        # Then slice: keep only the LAST k messages (dropping the oldest)
        # Example: if k=4 and we have 6 messages → keep messages[2:6]
        self.messages = self.messages[-self.k:]

    def clear(self) -> None:
        """Wipe all messages from history."""
        self.messages = []

print("BufferWindowMessageHistory class defined.")

In [ ]:
from langchain_core.runnables import ConfigurableFieldSpec

# We need a fresh chat_map for this section
chat_map_window = {}

def get_window_history(session_id: str, k: int = 4) -> BufferWindowMessageHistory:
    """Look up (or create) a window-based chat history for a given session."""
    if session_id not in chat_map_window:
        # Create a new history with the specified window size k
        chat_map_window[session_id] = BufferWindowMessageHistory(k=k)
    return chat_map_window[session_id]

print("get_window_history function defined.")

In [ ]:
# Rebuild the prompt + pipeline (same as Section 1)
# (Redefining here so this section is self-contained)
prompt_template_w = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template("You are a helpful assistant called Zeta."),
    MessagesPlaceholder(variable_name="history"),
    HumanMessagePromptTemplate.from_template("{query}"),
])
pipeline_w = prompt_template_w | llm

# Wrap with history — this time we pass EXTRA config fields for session_id AND k
# ConfigurableFieldSpec lets us pass custom parameters through config={"configurable": {...}}
pipeline_window = RunnableWithMessageHistory(
    pipeline_w,
    get_session_history=get_window_history,
    input_messages_key="query",
    history_messages_key="history",
    history_factory_config=[
        ConfigurableFieldSpec(
            id="session_id",
            annotation=str,
            name="Session ID",
            description="Unique identifier for this conversation session",
            default="default",
        ),
        ConfigurableFieldSpec(
            id="k",
            annotation=int,
            name="Window size k",
            description="How many messages to keep in the sliding window",
            default=4,
        )
    ]
)

print("pipeline_window (RunnableWithMessageHistory) created.")

In [ ]:
# ============================================================
# Demo A: Small window (k=4) — AI will FORGET our name
# ============================================================

# Note: k=4 means 4 message slots (2 turns × 2 messages each)

# Turn 1: Introduce ourselves
pipeline_window.invoke(
    {"query": "Hi, my name is Zariya!"},
    config={"configurable": {"session_id": "small_k", "k": 4}}
)

# Turns 2-4: Have a longer conversation (these will push Turn 1 out of the window)
for msg in [
    "What is LCEL in LangChain?",
    "What does the pipe operator | do in a chain?",
    "Can you give a one-line example of chaining a prompt with an LLM?",
]:
    pipeline_window.invoke(
        {"query": msg},
        config={"configurable": {"session_id": "small_k", "k": 4}}
    )

print("Conversation history after 4 turns (k=4):")
for msg in chat_map_window["small_k"].messages:
    role = "Human" if msg.__class__.__name__ == "HumanMessage" else "AI"
    print(f"  [{role}]: {msg.content[:80]}..." if len(msg.content) > 80 else f"  [{role}]: {msg.content}")

print("\n--- Notice: the first 'Hi my name is Zariya' message has been dropped! ---")

In [ ]:
# Now ask the AI our name — it should NOT remember (it's outside the window)
r_forget = pipeline_window.invoke(
    {"query": "What is my name?"},
    config={"configurable": {"session_id": "small_k", "k": 4}}
)
print("AI (k=4, name should be forgotten):", r_forget.content)

In [ ]:
# ============================================================
# Demo B: Large window (k=10) — AI WILL remember our name
# ============================================================

# Turn 1: Introduce ourselves
pipeline_window.invoke(
    {"query": "Hi, my name is Zariya!"},
    config={"configurable": {"session_id": "large_k", "k": 10}}
)

# Turns 2-4: Same conversation
for msg in [
    "What is LCEL in LangChain?",
    "What does the pipe operator | do in a chain?",
    "Can you give a one-line example of chaining a prompt with an LLM?",
]:
    pipeline_window.invoke(
        {"query": msg},
        config={"configurable": {"session_id": "large_k", "k": 10}}
    )

print(f"Total messages stored (k=10): {len(chat_map_window['large_k'].messages)}")
print("First message still in history:", chat_map_window["large_k"].messages[0].content)

In [ ]:
# Now ask the AI our name — it SHOULD remember (still within the window)
r_remember = pipeline_window.invoke(
    {"query": "What is my name?"},
    config={"configurable": {"session_id": "large_k", "k": 10}}
)
print("AI (k=10, name should be remembered):", r_remember.content)

In [ ]:
# ============================================================
# Summary: Compare message counts between the two sessions
# ============================================================
print("=" * 50)
print("COMPARISON: Buffer vs Window Memory")
print("=" * 50)
print(f"small_k (k=4) — messages in memory : {len(chat_map_window['small_k'].messages)}")
print(f"large_k (k=10) — messages in memory: {len(chat_map_window['large_k'].messages)}")
print()
print("With k=4 : AI forgets old messages (saves tokens, loses context)")
print("With k=10: AI remembers more (more tokens per request, more context)")
print()
print("Choosing k is a trade-off between: memory vs. cost & latency")

---
### Practice Exercise 2

Test your understanding of `ConversationBufferWindowMemory` by completing the tasks below.

**Tasks:**
1. Create a session `"experiment"` with `k=6`. Tell the AI:
   - Your name
   - Your university (UAF)
   - Your favorite subject
2. Have 3 more turns of any conversation (ask anything you like).
3. Ask the AI: *"What do you know about me?"* — check which facts it still remembers.
4. Increase `k` to `12` in a new session `"experiment_large"` and repeat the same steps. Compare the results.

**Challenge:** Can you modify `BufferWindowMessageHistory` to print a warning message whenever a message gets dropped from the window? (Hint: add a print inside `add_messages` when `len(self.messages) > self.k` before trimming.)

**Reflection questions (answer in a new markdown cell):**
- How does `k` relate to the number of conversation *turns* vs individual messages?
- What value of `k` would you choose for a customer support chatbot where conversations typically last 6-10 turns?

In [ ]:
# --- YOUR CODE HERE ---

# Task 1: Create 'experiment' session with k=6 and introduce yourself


# Task 2: Have 3 more turns of conversation


# Task 3: Ask what the AI knows about you


# Task 4: Repeat with k=12 in 'experiment_large'


# Challenge: Modify BufferWindowMessageHistory to print dropped messages


---
## Summary

| Memory Type | Keeps | Forgets | Best For |
|-------------|-------|---------|----------|
| `ConversationBufferMemory` | Everything | Nothing | Short conversations, debugging |
| `ConversationBufferWindowMemory` | Last `k` messages | Older messages | Long conversations, cost-sensitive apps |

Both are implemented with the same pattern:

```python
pipeline = prompt_template | llm
pipeline_with_history = RunnableWithMessageHistory(
    pipeline,
    get_session_history=<your_function>,
    input_messages_key="query",
    history_messages_key="history"
)
pipeline_with_history.invoke(
    {"query": "..."},
    config={"configurable": {"session_id": "..."}}
)
```

The only difference is **what kind of history object** your `get_session_history` function returns:
- Buffer: return `InMemoryChatMessageHistory()`
- Window: return `BufferWindowMessageHistory(k=<your_k>)`

---
**Next Step: Step 3 — Memory (Persistent / External Storage)**